In [ ]:
# ============================================================
# 01 — CHARGEMENT DE LA BASE ANALYTIQUE
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

# Répertoire des données préparées
PROCESSED_DIR = Path("../data/processed")

MODEL_DATA_FILE = PROCESSED_DIR / "model_data.parquet"

# Chargement
model_data = pd.read_parquet(MODEL_DATA_FILE)

print("model_data chargé avec succès.\n")

print("Dimensions :", model_data.shape)
print("Nombre de transactions uniques :", model_data["id"].nunique())

print("\nRépartition de la cible :")
print(
    model_data["is_fraud"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 02 — INVENTAIRE DES COLONNES
# ============================================================

print("=== COLONNES DE MODEL_DATA ===\n")

for i, col in enumerate(model_data.columns, start=1):
    print(f"{i:02d}. {col}")

In [ ]:
print("\n=== TYPES DE DONNÉES ===\n")

display(
    model_data.dtypes.to_frame("dtype")
)

In [ ]:
print("\n=== CARDINALITÉ DES VARIABLES CATÉGORIELLES ===\n")

categorical_candidates = [
    col
    for col in model_data.columns
    if model_data[col].dtype == "object"
]

cardinality = pd.DataFrame({
    "feature": categorical_candidates,
    "unique_values": [
        model_data[col].nunique(dropna=True)
        for col in categorical_candidates
    ]
})

display(
    cardinality.sort_values(
        "unique_values",
        ascending=False
    )
)

In [ ]:
# ============================================================
# 03 — CLASSIFICATION DES VARIABLES
# ============================================================

# ------------------------------------------------------------
# Cible
# ------------------------------------------------------------

TARGET = "is_fraud"

# ------------------------------------------------------------
# Identifiants
# ------------------------------------------------------------

ID_FEATURES = [
    "id",
    "client_id",
    "card_id",
    "merchant_id"
]

# ------------------------------------------------------------
# Dates / identifiants temporels techniques
# ------------------------------------------------------------

DATETIME_FEATURES = [
    "date",
    "previous_transaction_time"
]

# ------------------------------------------------------------
# Variables catégorielles
# ------------------------------------------------------------

CATEGORICAL_FEATURES = [
    "use_chip",
    "merchant_city",
    "merchant_state",
    "card_brand",
    "card_type",
    "has_chip",
    "card_on_dark_web"
]

# ------------------------------------------------------------
# Variables catégorielles à haute cardinalité
# ------------------------------------------------------------

HIGH_CARDINALITY_FEATURES = [
    "merchant_city"
]

# ------------------------------------------------------------
# Variables numériques candidates
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    "amount_numeric",
    "zip",
    "mcc",
    "has_error",
    "current_age",
    "yearly_income_numeric",
    "total_debt_numeric",
    "credit_score",
    "num_credit_cards",
    "credit_limit_numeric",
    "previous_amount",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "client_transaction_count_24h",
    "client_transaction_count_1h",
    "client_amount_24h",
    "amount_abs",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h",
    "year",
    "month",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "debt_to_income",
    "amount_to_income",
    "amount_to_credit_limit",
    "bad_pin",
    "bad_cvv",
    "bad_card_number",
    "bad_expiration",
    "bad_zipcode",
    "insufficient_balance",
    "technical_glitch",
    "error_count"
]

print("=== GROUPES DE VARIABLES ===")

print("\nIdentifiants :")
print(ID_FEATURES)

print("\nDates :")
print(DATETIME_FEATURES)

print("\nCatégorielles :")
print(CATEGORICAL_FEATURES)

print("\nNumériques :")
print(f"{len(NUMERIC_FEATURES)} variables")

In [ ]:
# Vérifier que toutes les variables existent réellement
all_candidate_features = (
    ID_FEATURES
    + DATETIME_FEATURES
    + CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + [TARGET]
)

missing_features = [
    col
    for col in all_candidate_features
    if col not in model_data.columns
]

duplicates_in_groups = (
    pd.Series(all_candidate_features)
    .value_counts()
)

print("\n=== CONTRÔLE ===")

print(
    "Variables absentes :",
    missing_features
)

print("\nVariables présentes dans plusieurs groupes :")

display(
    duplicates_in_groups[
        duplicates_in_groups > 1
    ]
)

In [ ]:
# ============================================================
# 04 — DÉTECTION DES VARIABLES CONSTANTES
# ============================================================

# Variables candidates hors cible
candidate_features = [
    col
    for col in model_data.columns
    if col != TARGET
]

# Nombre de valeurs distinctes par variable
nunique_features = (
    model_data[candidate_features]
    .nunique(dropna=False)
    .sort_values()
)

constant_features = (
    nunique_features[
        nunique_features <= 1
    ]
    .index
    .tolist()
)

print("=== VARIABLES CONSTANTES ===\n")

print(constant_features)

print(
    f"\nNombre de variables constantes : "
    f"{len(constant_features)}"
)

In [ ]:
print("\n=== DÉTAIL DES VARIABLES CONSTANTES ===")

for feature in constant_features:
    print(f"\n{feature}")
    print(
        model_data[feature]
        .value_counts(dropna=False)
    )

In [ ]:
# ============================================================
# 05 — SUPPRESSION DES VARIABLES INUTILES
# ============================================================

DROP_FEATURES = [
    "card_on_dark_web",  # variable constante
    "amount"             # version texte, remplacée par amount_numeric
]

# Vérifier que les colonnes existent avant suppression
DROP_FEATURES = [
    col
    for col in DROP_FEATURES
    if col in model_data.columns
]

model_data = model_data.drop(
    columns=DROP_FEATURES
)

print("Variables supprimées :")
print(DROP_FEATURES)

print("\nDimensions après suppression :")
print(model_data.shape)

print("\nColonnes restantes :")
print(model_data.columns.tolist())

In [ ]:
# ============================================================
# 06 — VALEURS INFINIES ET VALEURS MANQUANTES
# ============================================================

# ------------------------------------------------------------
# 1. Remplacer les valeurs infinies par NaN
# ------------------------------------------------------------

numeric_columns = model_data.select_dtypes(
    include=[np.number]
).columns

infinite_count_before = np.isinf(
    model_data[numeric_columns]
).sum().sum()

print("Valeurs infinies avant nettoyage :", infinite_count_before)

model_data[numeric_columns] = (
    model_data[numeric_columns]
    .replace([np.inf, -np.inf], np.nan)
)

infinite_count_after = np.isinf(
    model_data[numeric_columns]
).sum().sum()

print(
    "Valeurs infinies après nettoyage :",
    infinite_count_after
)


# ------------------------------------------------------------
# 2. Diagnostic des valeurs manquantes
# ------------------------------------------------------------

missing_summary = (
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = (
    missing_summary[missing_summary > 0]
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(model_data)
    * 100
)

print("\n=== VALEURS MANQUANTES ===")

display(missing_summary)

In [ ]:
# ============================================================
# 07 — CLASSIFICATION DES VALEURS MANQUANTES
# ============================================================

MISSING_AS_ZERO = [
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h"
]

MISSING_AS_NO_ERROR = [
    "errors"
]

MISSING_AS_CATEGORY = [
    "merchant_state"
]

MISSING_KEEP_NAN = [
    "previous_amount",
    "previous_transaction_time",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "amount_to_credit_limit",
    "zip"
]

missing_strategy = pd.DataFrame({
    "feature": (
        MISSING_AS_ZERO
        + MISSING_AS_NO_ERROR
        + MISSING_AS_CATEGORY
        + MISSING_KEEP_NAN
    ),
    "strategy": (
        ["zero"] * len(MISSING_AS_ZERO)
        + ["no_error"] * len(MISSING_AS_NO_ERROR)
        + ["category"] * len(MISSING_AS_CATEGORY)
        + ["keep_nan"] * len(MISSING_KEEP_NAN)
    )
})

print("=== STRATÉGIE DES VALEURS MANQUANTES ===")

display(missing_strategy)

In [ ]:
# Vérification : chaque variable ne doit apparaître qu'une seule fois

duplicates = (
    missing_strategy["feature"]
    .value_counts()
)

print("\nVariables présentes plusieurs fois :")

display(
    duplicates[duplicates > 1]
)

print(
    "\nNombre de variables classifiées :",
    missing_strategy["feature"].nunique()
)

In [ ]:
# ============================================================
# 08 — APPLICATION DES STRATÉGIES DE VALEURS MANQUANTES
# ============================================================

# ------------------------------------------------------------
# 1. Fenêtres comportementales sans historique
# ------------------------------------------------------------

for col in MISSING_AS_ZERO:
    model_data[col] = model_data[col].fillna(0)


# ------------------------------------------------------------
# 2. Absence d'erreur
# ------------------------------------------------------------

model_data["errors"] = (
    model_data["errors"]
    .fillna("No Error")
)


# ------------------------------------------------------------
# 3. Catégorie pour merchant_state
# ------------------------------------------------------------

model_data["merchant_state"] = (
    model_data["merchant_state"]
    .fillna("Unknown")
)


# ------------------------------------------------------------
# 4. Vérification
# ------------------------------------------------------------

print("=== VALEURS MANQUANTES APRÈS TRAITEMENT ===\n")

missing_after = (
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_after[missing_after > 0]
)

In [ ]:
# ============================================================
# 09 — TRAITEMENT DES VALEURS MANQUANTES RESTANTES
# ============================================================

# ------------------------------------------------------------
# 1. Historique du client
# ------------------------------------------------------------
# Pour les premières transactions d'un client, il n'existe pas
# encore de transaction précédente.
#
# On conserve donc une valeur neutre pour les variables
# comportementales numériques et une valeur indicatrice pour
# le temps.

model_data["previous_amount_missing"] = (
    model_data["previous_amount"].isna().astype("int8")
)

model_data["previous_amount"] = (
    model_data["previous_amount"].fillna(0)
)

model_data["amount_difference"] = (
    model_data["amount_difference"].fillna(0)
)

model_data["client_previous_avg_amount"] = (
    model_data["client_previous_avg_amount"].fillna(0)
)

model_data["amount_deviation_from_client"] = (
    model_data["amount_deviation_from_client"].fillna(0)
)

model_data["amount_change_ratio"] = (
    model_data["amount_change_ratio"].fillna(0)
)

# Temps depuis la transaction précédente :
# -1 indique qu'il n'existe pas de transaction précédente.
model_data["time_since_previous_transaction"] = (
    model_data["time_since_previous_transaction"]
    .fillna(-1)
)


# ------------------------------------------------------------
# 2. Date de la transaction précédente
# ------------------------------------------------------------
# Cette variable est utile pour les calculs historiques mais
# ne sera pas envoyée directement au modèle.
#
# On laisse donc les NaT pour le moment.

# previous_transaction_time reste inchangée


# ------------------------------------------------------------
# 3. Code postal
# ------------------------------------------------------------
# Le code postal est géographique et sa valeur 0 n'a pas de
# signification métier évidente.
#
# On crée donc un indicateur d'absence puis on utilise -1
# comme valeur technique.

model_data["zip_missing"] = (
    model_data["zip"].isna().astype("int8")
)

model_data["zip"] = (
    model_data["zip"].fillna(-1)
)


# ------------------------------------------------------------
# 4. Ratio montant / limite de crédit
# ------------------------------------------------------------
# Un ratio non calculable est différent d'un ratio nul.
# On conserve une indication de l'absence puis on utilise
# 0 comme valeur technique.

model_data["amount_to_credit_limit_missing"] = (
    model_data["amount_to_credit_limit"]
    .isna()
    .astype("int8")
)

model_data["amount_to_credit_limit"] = (
    model_data["amount_to_credit_limit"]
    .fillna(0)
)


# ------------------------------------------------------------
# 5. Contrôle
# ------------------------------------------------------------

remaining_missing = (
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("=== VALEURS MANQUANTES APRÈS TRAITEMENT ===")

display(
    remaining_missing[
        remaining_missing > 0
    ]
)

In [ ]:
# ============================================================
# 10 — DÉTECTION DES VALEURS EXTRÊMES
# ============================================================

outlier_features = [
    "amount_numeric",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "client_amount_24h",
    "client_absolute_amount_24h",
    "card_amount_24h",
    "merchant_amount_24h",
    "debt_to_income",
    "amount_to_income",
    "amount_to_credit_limit"
]

# Garder uniquement les variables présentes
outlier_features = [
    col
    for col in outlier_features
    if col in model_data.columns
]

# ------------------------------------------------------------
# Statistiques utiles pour détecter les extrêmes
# ------------------------------------------------------------

outlier_summary = []

for feature in outlier_features:

    series = model_data[feature].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        ((series < lower_bound) | (series > upper_bound))
        .sum()
    )

    outlier_summary.append({
        "feature": feature,
        "q1": q1,
        "median": series.median(),
        "q3": q3,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_pct": (
            outlier_count / len(series) * 100
        )
    })

outlier_summary = pd.DataFrame(
    outlier_summary
).sort_values(
    "outlier_pct",
    ascending=False
)

print("=== VALEURS EXTRÊMES ===")

display(outlier_summary)

In [ ]:
# ============================================================
# 11 — TRANSFORMATIONS ROBUSTES DES VARIABLES ASYMÉTRIQUES
# ============================================================

# ------------------------------------------------------------
# Transformation logarithmique standard
# ------------------------------------------------------------

def safe_log1p(series):
    """
    log(1 + x) pour x >= 0.
    Les valeurs négatives deviennent NaN.
    """
    return np.log1p(
        series.where(series >= 0)
    )


# ------------------------------------------------------------
# Transformation logarithmique signée
# ------------------------------------------------------------

def signed_log1p(series):
    """
    Transformation signée :
    sign(x) * log(1 + |x|)
    """
    return (
        np.sign(series)
        * np.log1p(series.abs())
    )


# ------------------------------------------------------------
# Montant
# ------------------------------------------------------------

model_data["amount_log"] = signed_log1p(
    model_data["amount_numeric"]
)

model_data["amount_difference_log"] = (
    safe_log1p(
        model_data["amount_difference"]
    )
)

# ------------------------------------------------------------
# Ratios très asymétriques
# ------------------------------------------------------------

model_data["amount_change_ratio_log"] = (
    safe_log1p(
        model_data["amount_change_ratio"]
    )
)

model_data["amount_deviation_from_client_log"] = (
    safe_log1p(
        model_data["amount_deviation_from_client"]
    )
)

model_data["amount_to_income_log"] = (
    signed_log1p(
        model_data["amount_to_income"]
    )
)

model_data["amount_to_credit_limit_log"] = (
    signed_log1p(
        model_data["amount_to_credit_limit"]
    )
)

# ------------------------------------------------------------
# Montants historiques
# ------------------------------------------------------------

model_data["client_amount_24h_log"] = (
    signed_log1p(
        model_data["client_amount_24h"]
    )
)

model_data["client_absolute_amount_24h_log"] = (
    safe_log1p(
        model_data["client_absolute_amount_24h"]
    )
)

model_data["card_amount_24h_log"] = (
    signed_log1p(
        model_data["card_amount_24h"]
    )
)

model_data["merchant_amount_24h_log"] = (
    signed_log1p(
        model_data["merchant_amount_24h"]
    )
)

print("Transformations logarithmiques créées.")

In [ ]:
# # ============================================================
# # 12 — SÉLECTION INITIALE DES FEATURES CANDIDATES
# # ============================================================

# TARGET = "is_fraud"

# # ------------------------------------------------------------
# # Variables à exclure du modèle
# # ------------------------------------------------------------

# EXCLUDE_FROM_MODEL = [
#     "id",                       # identifiant transaction
#     "date",                     # datetime brut
#     "previous_transaction_time",# datetime brut
#     "amount",                   # version texte
#     "errors",                   # remplacée par les indicateurs d'erreurs
# ]

# # Les identifiants client/carte/marchand ne sont pas encore
# # utilisés comme variables numériques brutes.
# # Ils seront conservés dans model_data pour d'éventuels
# # regroupements / contrôles, mais ne seront pas donnés
# # directement à XGBoost dans la première version.

# ID_CONTEXT_FEATURES = [
#     "client_id",
#     "card_id",
#     "merchant_id"
# ]

# # ------------------------------------------------------------
# # Variables catégorielles candidates
# # ------------------------------------------------------------

# CATEGORICAL_FEATURES = [
#     "use_chip",
#     "merchant_state",
#     "card_brand",
#     "card_type",
#     "has_chip"
# ]

# # merchant_city possède 12 173 modalités :
# # nous ne l'envoyons pas directement dans un One-Hot Encoding.
# HIGH_CARDINALITY_FEATURES = [
#     "merchant_city"
# ]

# # ------------------------------------------------------------
# # Variables numériques candidates
# # ------------------------------------------------------------

# NUMERIC_FEATURES = [
#     "amount_numeric",
#     "zip",
#     "mcc",
#     "has_error",
#     "current_age",
#     "yearly_income_numeric",
#     "total_debt_numeric",
#     "credit_score",
#     "num_credit_cards",
#     "credit_limit_numeric",

#     "previous_amount",
#     "time_since_previous_transaction",
#     "amount_difference",
#     "amount_change_ratio",
#     "client_previous_avg_amount",
#     "amount_deviation_from_client",

#     "client_transaction_count_1h",
#     "client_transaction_count_24h",
#     "client_amount_24h",
#     "amount_abs",
#     "client_absolute_amount_24h",

#     "card_transaction_count_24h",
#     "card_amount_24h",

#     "merchant_transaction_count_24h",
#     "merchant_amount_24h",

#     "year",
#     "month",
#     "transaction_hour",
#     "day_of_week",
#     "is_weekend",
#     "is_night",

#     "debt_to_income",
#     "amount_to_income",
#     "amount_to_credit_limit",

#     "bad_pin",
#     "bad_cvv",
#     "bad_card_number",
#     "bad_expiration",
#     "bad_zipcode",
#     "insufficient_balance",
#     "technical_glitch",
#     "error_count",

#     # Versions transformées
#     "amount_log",
#     "amount_difference_log",
#     "amount_change_ratio_log",
#     "amount_deviation_from_client_log",
#     "amount_to_income_log",
#     "amount_to_credit_limit_log",
#     "client_amount_24h_log",
#     "client_absolute_amount_24h_log",
#     "card_amount_24h_log",
#     "merchant_amount_24h_log"
# ]

# # ------------------------------------------------------------
# # Contrôles
# # ------------------------------------------------------------

# all_candidate_features = (
#     NUMERIC_FEATURES
#     + CATEGORICAL_FEATURES
# )

# missing_candidates = [
#     col
#     for col in all_candidate_features
#     if col not in model_data.columns
# ]

# duplicates = (
#     pd.Series(all_candidate_features)
#     .value_counts()
# )

# duplicated_candidates = duplicates[
#     duplicates > 1
# ]

# print("=== CONTRÔLE DES FEATURES ===")

# print(
#     "Features numériques :",
#     len(NUMERIC_FEATURES)
# )

# print(
#     "Features catégorielles :",
#     len(CATEGORICAL_FEATURES)
# )

# print(
#     "Features candidates totales :",
#     len(all_candidate_features)
# )

# print(
#     "\nFeatures absentes :",
#     missing_candidates
# )

# print("\nFeatures présentes plusieurs fois :")

# display(duplicated_candidates)

In [ ]:
# ============================================================
# 13 — LISTE FINALE PROVISOIRE DES FEATURES
# ============================================================

TARGET = "is_fraud"

# ------------------------------------------------------------
# Variables catégorielles
# ------------------------------------------------------------

CATEGORICAL_FEATURES = [
    "use_chip",
    "merchant_state",
    "card_brand",
    "card_type",
    "has_chip",
    "mcc"
]

# ------------------------------------------------------------
# Variables numériques
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    # Transaction
    "amount_numeric",
    "has_error",
    "error_count",

    # Profil client
    "current_age",
    "yearly_income_numeric",
    "total_debt_numeric",
    "credit_score",
    "num_credit_cards",

    # Carte
    "credit_limit_numeric",

    # Historique client
    "previous_amount",
    "previous_amount_missing",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",

    # Vélocité / comportement client
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "amount_abs",
    "client_absolute_amount_24h",

    # Comportement carte
    "card_transaction_count_24h",
    "card_amount_24h",

    # Comportement marchand
    "merchant_transaction_count_24h",
    "merchant_amount_24h",

    # Temps
    "year",
    "month",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",

    # Ratios
    "debt_to_income",
    "amount_to_income",
    "amount_to_credit_limit",

    # Indicateur d'absence du ratio crédit
    "amount_to_credit_limit_missing",

    # Types d'erreurs
    "bad_pin",
    "bad_cvv",
    "bad_card_number",
    "bad_expiration",
    "bad_zipcode",
    "insufficient_balance",
    "technical_glitch",

    # Versions transformées
    "amount_log",
    "amount_difference_log",
    "amount_change_ratio_log",
    "amount_deviation_from_client_log",
    "amount_to_income_log",
    "amount_to_credit_limit_log",
    "client_amount_24h_log",
    "client_absolute_amount_24h_log",
    "card_amount_24h_log",
    "merchant_amount_24h_log",

    # Information manquante géographique
    "zip_missing"
]

# ------------------------------------------------------------
# Vérification
# ------------------------------------------------------------

FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

missing_features = [
    col
    for col in FEATURES
    if col not in model_data.columns
]

duplicates = (
    pd.Series(FEATURES)
    .value_counts()
)

print("=== FEATURES CANDIDATES ===")

print("Features numériques      :", len(NUMERIC_FEATURES))
print("Features catégorielles  :", len(CATEGORICAL_FEATURES))
print("Total                   :", len(FEATURES))

print("\nFeatures absentes :")
print(missing_features)

print("\nFeatures dupliquées :")
display(
    duplicates[duplicates > 1]
)

In [ ]:
# ============================================================
# 14 — SPLIT TEMPOREL TRAIN / VALIDATION / TEST
# ============================================================

# ------------------------------------------------------------
# Tri chronologique
# ------------------------------------------------------------

model_data = (
    model_data
    .sort_values(["date", "id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Création des ensembles
# ------------------------------------------------------------

train_data = model_data[
    model_data["year"] <= 2017
].copy()

validation_data = model_data[
    model_data["year"] == 2018
].copy()

test_data = model_data[
    model_data["year"] == 2019
].copy()

# ------------------------------------------------------------
# Fonction de résumé
# ------------------------------------------------------------

def split_summary(name, df):
    fraud_count = df["is_fraud"].sum()
    
    print(f"\n=== {name} ===")
    print("Lignes :", len(df))
    print("Date min :", df["date"].min())
    print("Date max :", df["date"].max())
    print("Fraudes :", fraud_count)
    print(
        "Taux de fraude :",
        f"{df['is_fraud'].mean() * 100:.4f} %"
    )

# ------------------------------------------------------------
# Résultats
# ------------------------------------------------------------

split_summary("TRAIN", train_data)
split_summary("VALIDATION", validation_data)
split_summary("TEST", test_data)

In [ ]:
# ------------------------------------------------------------
# Vérification qu'il n'y a aucun chevauchement
# ------------------------------------------------------------

train_ids = set(train_data["id"])
validation_ids = set(validation_data["id"])
test_ids = set(test_data["id"])

print("\n=== VÉRIFICATION DES CHEVAUCHEMENTS ===")

print(
    "Train ∩ Validation :",
    len(train_ids & validation_ids)
)

print(
    "Train ∩ Test       :",
    len(train_ids & test_ids)
)

print(
    "Validation ∩ Test  :",
    len(validation_ids & test_ids)
)

In [ ]:
# ============================================================
# 15 — DISTRIBUTION DE LA CIBLE PAR ENSEMBLE
# ============================================================

split_target_summary = pd.DataFrame({
    "train": train_data["is_fraud"].value_counts(),
    "validation": validation_data["is_fraud"].value_counts(),
    "test": test_data["is_fraud"].value_counts()
}).fillna(0).astype(int)

print("=== NOMBRE DE TRANSACTIONS PAR CLASSE ===")

display(split_target_summary)

print("\n=== POURCENTAGE DE FRAUDE ===")

split_fraud_rates = pd.DataFrame({
    "train": [train_data["is_fraud"].mean() * 100],
    "validation": [validation_data["is_fraud"].mean() * 100],
    "test": [test_data["is_fraud"].mean() * 100]
}, index=["fraud_rate"])

display(
    split_fraud_rates.round(4)
)

In [ ]:
print("\n=== CHRONOLOGIE ===")

print(
    "Fin du train        :",
    train_data["date"].max()
)

print(
    "Début validation    :",
    validation_data["date"].min()
)

print(
    "Fin validation      :",
    validation_data["date"].max()
)

print(
    "Début test          :",
    test_data["date"].min()
)

In [ ]:
# ============================================================
# 16 — SÉPARATION DES FEATURES ET DE LA CIBLE
# ============================================================

# ------------------------------------------------------------
# Features catégorielles et numériques
# ------------------------------------------------------------

X_train = train_data[FEATURES].copy()
y_train = train_data[TARGET].copy()

X_validation = validation_data[FEATURES].copy()
y_validation = validation_data[TARGET].copy()

X_test = test_data[FEATURES].copy()
y_test = test_data[TARGET].copy()


# ------------------------------------------------------------
# Contrôles
# ------------------------------------------------------------

print("=== DIMENSIONS ===")

print("X_train      :", X_train.shape)
print("y_train      :", y_train.shape)

print("X_validation :", X_validation.shape)
print("y_validation :", y_validation.shape)

print("X_test       :", X_test.shape)
print("y_test       :", y_test.shape)


print("\n=== DISTRIBUTION DE LA CIBLE ===")

print("Train :")
print(y_train.value_counts().sort_index())

print("\nValidation :")
print(y_validation.value_counts().sort_index())

print("\nTest :")
print(y_test.value_counts().sort_index())

In [ ]:
# ============================================================
# 17 — ENCODAGE DES VARIABLES CATÉGORIELLES
# ============================================================

from sklearn.preprocessing import OneHotEncoder

# Variables catégorielles retenues
CATEGORICAL_FEATURES = [
    "use_chip",
    "merchant_state",
    "card_brand",
    "card_type",
    "has_chip",
    "mcc"
]

# Création de l'encodeur
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Apprentissage uniquement sur le train
categorical_encoder.fit(
    X_train[CATEGORICAL_FEATURES]
)

print("Encodeur entraîné sur le train.")

print(
    "\nNombre de variables catégorielles d'origine :",
    len(CATEGORICAL_FEATURES)
)

print(
    "Nombre de modalités encodées :",
    len(categorical_encoder.get_feature_names_out())
)

In [ ]:
print("\n=== MODALITÉS APPRISES ===")

for feature, categories in zip(
    CATEGORICAL_FEATURES,
    categorical_encoder.categories_
):
    print(
        f"{feature:20s} : {len(categories)} modalités"
    )

In [ ]:
# ============================================================
# 18 — TRANSFORMATION DES VARIABLES CATÉGORIELLES
# ============================================================

X_train_cat = categorical_encoder.transform(
    X_train[CATEGORICAL_FEATURES]
)

X_validation_cat = categorical_encoder.transform(
    X_validation[CATEGORICAL_FEATURES]
)

X_test_cat = categorical_encoder.transform(
    X_test[CATEGORICAL_FEATURES]
)

print("=== MATRICES CATÉGORIELLES ===")

print(
    "X_train_cat      :",
    X_train_cat.shape
)

print(
    "X_validation_cat :",
    X_validation_cat.shape
)

print(
    "X_test_cat       :",
    X_test_cat.shape
)

In [ ]:
print("\nNombre de colonnes encodées :")

print(
    "Train      :", X_train_cat.shape[1]
)

print(
    "Validation :", X_validation_cat.shape[1]
)

print(
    "Test       :", X_test_cat.shape[1]
)

In [ ]:
# ============================================================
# 19 — PRÉPARATION DES VARIABLES NUMÉRIQUES
# ============================================================

# ------------------------------------------------------------
# Variables numériques finales
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    col
    for col in NUMERIC_FEATURES
    if col in model_data.columns
]

# ------------------------------------------------------------
# Extraction des trois ensembles
# ------------------------------------------------------------

X_train_num = train_data[NUMERIC_FEATURES].copy()
X_validation_num = validation_data[NUMERIC_FEATURES].copy()
X_test_num = test_data[NUMERIC_FEATURES].copy()

# ------------------------------------------------------------
# Sécurité : remplacer les valeurs infinies
# ------------------------------------------------------------

for df in [
    X_train_num,
    X_validation_num,
    X_test_num
]:
    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

# ------------------------------------------------------------
# Contrôles
# ------------------------------------------------------------

print("=== DIMENSIONS ===")

print("X_train_num      :", X_train_num.shape)
print("X_validation_num :", X_validation_num.shape)
print("X_test_num       :", X_test_num.shape)

print("\n=== TYPES ===")

print(
    X_train_num.dtypes
    .value_counts()
)

print("\n=== VALEURS MANQUANTES ===")

numeric_missing = pd.DataFrame({
    "train": X_train_num.isna().sum(),
    "validation": X_validation_num.isna().sum(),
    "test": X_test_num.isna().sum()
})

display(
    numeric_missing[
        numeric_missing.sum(axis=1) > 0
    ]
)

In [ ]:
# ============================================================
# 20 — CONTRÔLE DE COHÉRENCE DES ENSEMBLES
# ============================================================

# ------------------------------------------------------------
# Vérification des colonnes
# ------------------------------------------------------------

print("=== COLONNES NUMÉRIQUES ===")

print(
    "Train = Validation :",
    list(X_train_num.columns) == list(X_validation_num.columns)
)

print(
    "Train = Test       :",
    list(X_train_num.columns) == list(X_test_num.columns)
)

# ------------------------------------------------------------
# Vérification des types
# ------------------------------------------------------------

print("\n=== TYPES ===")

print(
    "Train = Validation :",
    (
        X_train_num.dtypes.values
        == X_validation_num.dtypes.values
    ).all()
)

print(
    "Train = Test       :",
    (
        X_train_num.dtypes.values
        == X_test_num.dtypes.values
    ).all()
)

# ------------------------------------------------------------
# Résumé des dimensions
# ------------------------------------------------------------

print("\n=== RÉSUMÉ ===")

print("Variables numériques :", len(NUMERIC_FEATURES))
print("Variables catégorielles encodées :", X_train_cat.shape[1])

print(
    "Features totales après assemblage :",
    X_train_num.shape[1] + X_train_cat.shape[1]
)

In [ ]:
# ============================================================
# 21 — ASSEMBLAGE DES MATRICES FINALES
# ============================================================

from scipy import sparse

# ------------------------------------------------------------
# Conversion des variables numériques en matrices sparse
# ------------------------------------------------------------

X_train_num_sparse = sparse.csr_matrix(
    X_train_num.to_numpy(dtype=np.float32)
)

X_validation_num_sparse = sparse.csr_matrix(
    X_validation_num.to_numpy(dtype=np.float32)
)

X_test_num_sparse = sparse.csr_matrix(
    X_test_num.to_numpy(dtype=np.float32)
)

# ------------------------------------------------------------
# Assemblage numérique + catégoriel
# ------------------------------------------------------------

X_train_final = sparse.hstack(
    [
        X_train_num_sparse,
        X_train_cat
    ],
    format="csr"
)

X_validation_final = sparse.hstack(
    [
        X_validation_num_sparse,
        X_validation_cat
    ],
    format="csr"
)

X_test_final = sparse.hstack(
    [
        X_test_num_sparse,
        X_test_cat
    ],
    format="csr"
)

print("=== MATRICES FINALES ===")

print(
    "X_train_final      :",
    X_train_final.shape
)

print(
    "X_validation_final :",
    X_validation_final.shape
)

print(
    "X_test_final       :",
    X_test_final.shape
)

In [ ]:
print("\n=== CONTRÔLE ===")

print(
    "Features train      :",
    X_train_final.shape[1]
)

print(
    "Features validation :",
    X_validation_final.shape[1]
)

print(
    "Features test       :",
    X_test_final.shape[1]
)

print(
    "\nFormat train :",
    type(X_train_final).__name__
)

print(
    "Format validation :",
    type(X_validation_final).__name__
)

print(
    "Format test :",
    type(X_test_final).__name__
)

In [ ]:
# ============================================================
# 22 — DÉSÉQUILIBRE DES CLASSES POUR XGBOOST
# ============================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = (
    negative_count / positive_count
)

print("=== DÉSÉQUILIBRE DU TRAIN ===")

print(
    "Classe 0 (non fraude) :",
    negative_count
)

print(
    "Classe 1 (fraude)     :",
    positive_count
)

print(
    "Ratio normal / fraude :",
    f"{scale_pos_weight:.2f}"
)

print(
    "\nTaux de fraude train :",
    f"{y_train.mean() * 100:.4f} %"
)

In [ ]:
# ============================================================
# 23 — CONTRÔLE FINAL AVANT ENTRAÎNEMENT
# ============================================================

print("=== MATRICES ===")

print("X_train_final      :", X_train_final.shape)
print("X_validation_final :", X_validation_final.shape)
print("X_test_final       :", X_test_final.shape)


print("\n=== CIBLES ===")

print("y_train :")
print(y_train.value_counts().sort_index())

print("\ny_validation :")
print(y_validation.value_counts().sort_index())

print("\ny_test :")
print(y_test.value_counts().sort_index())


print("\n=== VALEURS MANQUANTES ===")

print(
    "Train :",
    np.isnan(X_train_final.data).sum()
)

print(
    "Validation :",
    np.isnan(X_validation_final.data).sum()
)

print(
    "Test :",
    np.isnan(X_test_final.data).sum()
)


print("\n=== VALEURS INFINIES ===")

print(
    "Train :",
    np.isinf(X_train_final.data).sum()
)

print(
    "Validation :",
    np.isinf(X_validation_final.data).sum()
)

print(
    "Test :",
    np.isinf(X_test_final.data).sum()
)


print("\n=== PARAMÈTRE INITIAL XGBOOST ===")

print(
    "scale_pos_weight :",
    round(scale_pos_weight, 2)
)

In [ ]:
# ============================================================
# 24 — SAUVEGARDE DES DONNÉES PRÊTES POUR L'ENTRAÎNEMENT
# ============================================================

import joblib

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Matrices
joblib.dump(
    X_train_final,
    OUTPUT_DIR / "X_train_final.joblib"
)

joblib.dump(
    X_validation_final,
    OUTPUT_DIR / "X_validation_final.joblib"
)

joblib.dump(
    X_test_final,
    OUTPUT_DIR / "X_test_final.joblib"
)

# Cibles
joblib.dump(
    y_train,
    OUTPUT_DIR / "y_train.joblib"
)

joblib.dump(
    y_validation,
    OUTPUT_DIR / "y_validation.joblib"
)

joblib.dump(
    y_test,
    OUTPUT_DIR / "y_test.joblib"
)

# Paramètre de déséquilibre
joblib.dump(
    scale_pos_weight,
    OUTPUT_DIR / "scale_pos_weight.joblib"
)

print("Données d'entraînement sauvegardées avec succès.")

print("\nRépertoire :")
print(OUTPUT_DIR.resolve())

In [ ]:
# ============================================================
# 25 — SAUVEGARDE MÉMOIRE-EFFICACE DES MATRICES SPARSE
# ============================================================

from scipy import sparse

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sauvegarde native sparse
sparse.save_npz(
    OUTPUT_DIR / "X_train_final.npz",
    X_train_final
)

sparse.save_npz(
    OUTPUT_DIR / "X_validation_final.npz",
    X_validation_final
)

sparse.save_npz(
    OUTPUT_DIR / "X_test_final.npz",
    X_test_final
)

print("Matrices sparse sauvegardées en format .npz.")

print("\nFichiers créés :")
print(OUTPUT_DIR / "X_train_final.npz")
print(OUTPUT_DIR / "X_validation_final.npz")
print(OUTPUT_DIR / "X_test_final.npz")

In [ ]:
# ============================================================
# 26 — SAUVEGARDE DES CIBLES
# ============================================================

joblib.dump(
    y_train,
    OUTPUT_DIR / "y_train.joblib"
)

joblib.dump(
    y_validation,
    OUTPUT_DIR / "y_validation.joblib"
)

joblib.dump(
    y_test,
    OUTPUT_DIR / "y_test.joblib"
)

joblib.dump(
    scale_pos_weight,
    OUTPUT_DIR / "scale_pos_weight.joblib"
)

print("Cibles et paramètres sauvegardés.")

# 📊 27 — Visualisations du contrôle et de la préparation des données

Les graphiques ci-dessous documentent la préparation : répartition des ensembles temporels, déséquilibre de la cible, valeurs manquantes, valeurs extrêmes, encodage et densité des matrices. Ils utilisent les objets déjà construits dans ce notebook.


In [ ]:
# 1. Volume des ensembles temporels
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

split_sizes = pd.Series({
    "Train (≤ 2017)": len(train_data),
    "Validation (2018)": len(validation_data),
    "Test (2019)": len(test_data),
})
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(split_sizes.index, split_sizes.values)
ax.set_title("Taille des ensembles après découpage temporel")
ax.set_ylabel("Nombre de transactions")
ax.ticklabel_format(axis="y", style="plain")
for b, v in zip(bars, split_sizes.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:,.0f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# 2. Taux de fraude par ensemble temporel
fraud_rates = pd.Series({
    "Train (≤ 2017)": train_data["is_fraud"].mean()*100,
    "Validation (2018)": validation_data["is_fraud"].mean()*100,
    "Test (2019)": test_data["is_fraud"].mean()*100,
})
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(fraud_rates.index, fraud_rates.values)
ax.set_title("Comparaison du taux de fraude entre les ensembles")
ax.set_ylabel("Fraudes (%)")
for b, v in zip(bars, fraud_rates.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:.4f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# 3. Classes légitimes / fraude dans chaque ensemble
class_counts = pd.DataFrame({
    "Train": train_data["is_fraud"].value_counts().reindex([0,1], fill_value=0),
    "Validation": validation_data["is_fraud"].value_counts().reindex([0,1], fill_value=0),
    "Test": test_data["is_fraud"].value_counts().reindex([0,1], fill_value=0),
}, index=[0,1])
fig, ax = plt.subplots(figsize=(8, 4.5))
class_counts.T.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Nombre de transactions par classe et par ensemble")
ax.set_xlabel("Ensemble")
ax.set_ylabel("Nombre de transactions")
ax.legend(["Légitime (0)", "Fraude (1)"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Valeurs manquantes restantes par variable (si le résumé existe)
if "missing_summary" in globals() and isinstance(missing_summary, pd.DataFrame) and len(missing_summary):
    missing_plot = missing_summary.sort_values("missing_pct", ascending=True).tail(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(missing_plot.index.astype(str), missing_plot["missing_pct"])
    ax.set_title("Top 20 des variables avec le plus de valeurs manquantes")
    ax.set_xlabel("Valeurs manquantes (%)")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune valeur manquante résiduelle à représenter dans missing_summary.")


In [ ]:
# 5. Résumé des valeurs extrêmes (si calculé)
if "outlier_summary" in globals() and isinstance(outlier_summary, pd.DataFrame) and len(outlier_summary):
    print("Colonnes disponibles dans outlier_summary :", outlier_summary.columns.tolist())
    count_col = next((c for c in ["outlier_count", "n_outliers", "outliers", "count_outliers"] if c in outlier_summary.columns), None)
    if count_col:
        plot_data = outlier_summary.sort_values(count_col).tail(15)
        label_col = "feature" if "feature" in plot_data.columns else plot_data.columns[0]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(plot_data[label_col].astype(str), plot_data[count_col])
        ax.set_title("Variables avec le plus de valeurs extrêmes détectées")
        ax.set_xlabel("Nombre de valeurs extrêmes")
        plt.tight_layout()
        plt.show()
    else:
        print("Le résumé existe, mais ses colonnes ne permettent pas d'identifier automatiquement le nombre d'extrêmes.")


In [ ]:
# 6. Nombre de modalités des variables catégorielles
if "categorical_encoder" in globals() and "CATEGORICAL_FEATURES" in globals():
    cardinalities = pd.Series(
        [len(cats) for cats in categorical_encoder.categories_],
        index=CATEGORICAL_FEATURES
    ).sort_values()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.barh(cardinalities.index, cardinalities.values)
    ax.set_title("Nombre de modalités apprises par variable catégorielle")
    ax.set_xlabel("Nombre de modalités")
    plt.tight_layout()
    plt.show()


In [ ]:
# 7. Contribution des variables numériques et catégorielles au nombre de features
num_count = X_train_num.shape[1]
cat_count = X_train_cat.shape[1]
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["Numériques", "One-Hot catégorielles"], [num_count, cat_count])
ax.set_title("Composition des features après encodage")
ax.set_ylabel("Nombre de colonnes")
for b, v in zip(bars, [num_count, cat_count]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Densité des matrices sparse — sans conversion en matrice dense
from scipy import sparse

matrices = {
    "Train": X_train_final,
    "Validation": X_validation_final,
    "Test": X_test_final,
}
density_pct = {
    name: matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100
    for name, matrix in matrices.items()
}
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(list(density_pct.keys()), list(density_pct.values()))
ax.set_title("Densité des matrices finales sparse")
ax.set_ylabel("Cellules non nulles (%)")
for b, v in zip(bars, density_pct.values()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:.3f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()
